<a href="https://colab.research.google.com/github/manikantavs01/DeepLearning_Hackaton/blob/main/genai_support_ticket_classification_fewshot_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#importing libraries
import os
import pandas as pd
import ast
import time
import re

In [2]:
from getpass import getpass

key = getpass('Please enter your together AI API Key here: ')

Please enter your together AI API Key here: ··········


In [3]:
os.environ['GROQ_API_KEY'] = key

In [4]:
#installing groq
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.5 MB/s eta 0:00:00


In [5]:
#setting client and model
from groq import Groq
client=Groq()
model="llama3-70b-8192"

In [6]:
# Reding test and train data
test = pd.read_csv('test_genai.csv')
train = pd.read_csv('train_genai.csv')

In [7]:
#chat function
def get_response(prompt, model=model):
    messages = [{"role":"user","content":prompt}]
    client = Groq()
    response = client.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content

In [8]:
#Few shot examples
few_shot_examples = f''' You are an helpful customer support ticket classification AI assistant. Given a ticket your job is to classfiy it into one of the following categories
  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns&Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, General Inquiry
  type as one of Incident, Request, Change, problem
  priority as low, medium, high
  language as the language code for the email's language
  few examples are provided as few shot examples in the following format
  Strictly output only classification without any additional text'''

# Add the 8 labeled training emails as few-shot examples
for _, row in train.iterrows():
    few_shot_examples += f'Ticket: "{row["ticket_body"]}"\n' \
                       f'Department: {row["department"]}\n' \
                       f'Type: {row["type"]}\n' \
                       f'Priority: {row["priority"]}\n' \
                       f'Language: {row["language"]}\n\n'

In [9]:
# Function to classify test emails in batches
def classify_tickets(tickets):
    classified_output = []
    for email in tickets:
      input_prompt = few_shot_examples + f'Ticket: "{email}"\n' \
                                        f'Department:\nType:\nPriority:\nLanguage:'
      #print(input_prompt)
      try:
        response = get_response(input_prompt)
        #print(response)
        pattern = r"Department:\s*(.*?)\s*Type:\s*(.*?)\s*Priority:\s*(.*?)\s*Language:\s*(.*)"
        match = re.search(pattern, response)
        classified_output.append({
              "email": email,
              "department": match.group(1),
              "type": match.group(2),
              "priority": match.group(3),
              "language": match.group(4)
              })
      except Exception as e:
        classified_output.append({
            "email": email,
            "department": "Error",
            "type": "Error",
            "priority": "Error",
            "language": "Error"
              })
      #time.sleep(0.5) # Prevent API rate limit issues
    return(classified_output)

In [ ]:
# Specify the device
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

<module 'torch' from '/usr/local/lib/python3.11/dist-packages/torch/__init__.py'>

In [ ]:
# Run classification on the test emails
test_emails = test["ticket_body"].tolist()
classified_data = classify_tickets(test_emails)

In [12]:
!nvidia-smi

Sat Mar  1 15:51:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
classified_data=pd.DataFrame(classified_data)


NameError: name 'pd' is not defined

In [2]:
classified_data.shape

NameError: name 'classified_data' is not defined

In [126]:
classified_data.to_csv("classified_tickets.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets.csv'.")

Classification completed. Results saved to 'classified_tickets.csv'.
